In [1]:
import numpy as np
import json
import wget
from bs4 import BeautifulSoup
import tarfile
from astropy.table import Table
import os

In [4]:
base_url = 'https://s3.amazonaws.com/grizli-v2/JwstMosaics/v6/index.html'
output_directory = '/Users/marchuertascompany/Documents/data/astropile/cosweb/'  # Update this path as needed
field_identifier = 'ceers'

# make sure the output directory exists
try:
    os.chdir(output_directory)
    os.chdir('..')
except:
    print('output directory not found, made the dir.')
    os.mkdir(output_directory)


# Download the index.html file
file = wget.download(base_url)

# Read the content of the file
with open(file, "r") as f:
    file_content = f.read()

# Parse the HTML content
soup = BeautifulSoup(file_content, 'html.parser')

jwstfiles = []
for temp in soup.find_all('a'):
    if (field_identifier in temp['href']) and ('_sci' in temp['href']):
        jwstfiles.append(temp['href'])

# Print and download files
for url in jwstfiles:
    # Extract the filename from the URL
    filename = url.split('/')[-1]
    
    # Determine the filter from the filename
    temp = filename.split('_')[0].split('-')
    if temp[-1] != 'clear':
        filter_name = temp[-1]
    else:
        filter_name = temp[-2]

    # Print filter info and URL
    print('\nfilter: ', filter_name)
    print('url: ', url)

    # Construct the full local filepath
    full_local_path = os.path.join(output_directory, filename)

    # Download the file to the specified output directory
    wget.download(url, out=full_local_path)

# for the photometry table
for temp in soup.find_all('a'):
    if (field_identifier in temp['href']) & ('photoz' in temp['href']):
        print(temp['href'])
        photoz_url = temp['href']

# download the photoz file
filename = photoz_url.split('/')[-1]
full_local_path = os.path.join(output_directory, filename)
file = wget.download(photoz_url, out=output_directory)

# unzip the file
tar = tarfile.open(file)
tar.extractall(path=output_directory)
tar.close()

# read it in as a table
fnames = os.listdir(output_directory)
for fname in fnames:
    if 'eazypy.zout' in fname:
        phot_table = Table.read(output_directory + fname)
phot_table


filter:  f105w
url:  https://s3.amazonaws.com/grizli-v2/JwstMosaics/v6/cosweb-grizli-v6.0-f105w_drz_sci.fits.gz

filter:  f110w
url:  https://s3.amazonaws.com/grizli-v2/JwstMosaics/v6/cosweb-grizli-v6.0-f110w_drz_sci.fits.gz

filter:  f115w
url:  https://s3.amazonaws.com/grizli-v2/JwstMosaics/v6/cosweb-grizli-v6.0-f115w-clear_drc_sci.fits.gz


KeyboardInterrupt: 

'3.2'